# 🧬 Genetik Algoritmalar (Genetic Algorithms)

## Teori
Genetik Algoritmalar (GA), Charles Darwin'in doğal seçilim teorisinden ilham alan evrimsel optimizasyon yöntemleridir. 1970'lerde John Holland tarafından geliştirilmiştir.

### Temel Biyolojik Analoglar
| Biyoloji | GA Karşılığı |
|---|---|
| Birey | Çözüm adayı |
| Kromozom | Çözümün kodlanmış temsili |
| Gen | Çözümün bir parametresi |
| Popülasyon | Çözüm adayları kümesi |
| Uyum (Fitness) | Çözümün kalitesi |
| Seçilim | İyi çözümlerin seçilmesi |
| Çaprazlama | İki çözümün kombinasyonu |
| Mutasyon | Rastgele küçük değişiklik |

### GA'nın Adımları
```
1. Başlangıç popülasyonunu oluştur
2. Her bireyin fitness değerini hesapla
3. Durdurma kriteri sağlandıysa → DUR
4. Ebeveyn seçimi (Selection)
5. Çaprazlama (Crossover)
6. Mutasyon (Mutation)
7. Yeni popülasyonu oluştur → Adım 2'ye dön
```

In [ ]:
# ============================================================
# KÜTÜPHANE İMPORTLARI
# ============================================================
import numpy as np                    # Sayısal işlemler
import matplotlib.pyplot as plt       # Görselleştirme
import matplotlib.animation as animation  # Animasyon
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Tekrar üretilebilirlik için sabit seed
np.random.seed(42)
print("✅ Kütüphaneler yüklendi.")

## 1. Problem Tanımı

**Rastrigin Fonksiyonu** — Optimizasyon literatürünün klasik benchmark problemi.

$$f(x) = An + \sum_{i=1}^{n} \left[ x_i^2 - A\cos(2\pi x_i) \right]$$

- **Küresel minimum**: $f(0, 0, ..., 0) = 0$
- **Arama alanı**: $x_i \in [-5.12, 5.12]$
- **Zorluk**: Çok sayıda yerel minimum içerir → GA gibi global arama yöntemleri gerektirir

In [ ]:
# ============================================================
# FITNESS FONKSİYONU — RASTRIGIN
# ============================================================

def rastrigin(x):
    """
    Rastrigin optimizasyon fonksiyonu.
    
    Parametreler:
        x (np.ndarray): Çözüm vektörü, şekil (n_dims,)
    
    Döndürür:
        float: Fitness değeri (düşük = daha iyi)
    """
    A = 10                            # Rastrigin sabitesi
    n = len(x)                        # Boyut sayısı
    
    # Formül: An + Σ[xi² - A·cos(2π·xi)]
    return A * n + np.sum(x**2 - A * np.cos(2 * np.pi * x))


# Fonksiyonu 2D olarak görselleştirelim
x_range = np.linspace(-5.12, 5.12, 300)
y_range = np.linspace(-5.12, 5.12, 300)
X, Y = np.meshgrid(x_range, y_range)    # 300x300 grid

# Grid üzerindeki her nokta için Rastrigin değerini hesapla
Z = np.array([[rastrigin(np.array([X[i,j], Y[i,j]])) 
               for j in range(300)] for i in range(300)])

# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kontur haritası
axes[0].contourf(X, Y, Z, levels=50, cmap='viridis')
axes[0].contour(X, Y, Z, levels=20, colors='white', alpha=0.3, linewidths=0.5)
axes[0].scatter([0], [0], color='red', s=100, zorder=5, label='Global min (0,0)')
axes[0].set_title('Rastrigin Fonksiyonu — Kontur Haritası', fontsize=13)
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')
axes[0].legend()

# 3D yüzey grafiği
ax3d = fig.add_subplot(1, 2, 2, projection='3d') if False else axes[1]
# 2D kesit (x2=0 sabit)
x1_line = np.linspace(-5.12, 5.12, 500)
z_line  = [rastrigin(np.array([x, 0])) for x in x1_line]
axes[1].plot(x1_line, z_line, 'b-', linewidth=2)
axes[1].axvline(x=0, color='red', linestyle='--', label='Global min x=0')
axes[1].set_title('Rastrigin Kesiti (x₂=0)', fontsize=13)
axes[1].set_xlabel('x₁'); axes[1].set_ylabel('f(x₁, 0)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rastrigin_fonksiyonu.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Global minimum: f(0,0) = {rastrigin(np.array([0.0, 0.0])):.4f}")

## 2. GA Bileşenleri

### 2.1 Kodlama (Encoding)
Gerçek sayılı (real-valued) encoding kullanıyoruz. Her birey, $n$ boyutlu bir gerçek sayı vektörüdür.

In [ ]:
# ============================================================
# POPÜLASYON BAŞLATMA
# ============================================================

def initialize_population(pop_size, n_dims, lower_bound, upper_bound):
    """
    Rastgele başlangıç popülasyonu oluşturur.
    
    Parametreler:
        pop_size (int)    : Popülasyon büyüklüğü (birey sayısı)
        n_dims   (int)    : Problem boyutu (gen sayısı)
        lower_bound (float): Arama alanı alt sınırı
        upper_bound (float): Arama alanı üst sınırı
    
    Döndürür:
        np.ndarray: Şekil (pop_size, n_dims) — popülasyon matrisi
    """
    # Uniform dağılımdan rastgele bireyler örnekle
    # Her satır = bir birey, her sütun = bir gen/parametre
    population = np.random.uniform(
        low=lower_bound,
        high=upper_bound,
        size=(pop_size, n_dims)    # pop_size x n_dims matris
    )
    return population


def evaluate_population(population, fitness_fn):
    """
    Tüm bireylerin fitness değerini hesaplar.
    
    Parametreler:
        population (np.ndarray): Popülasyon matrisi (pop_size, n_dims)
        fitness_fn (callable)  : Fitness fonksiyonu
    
    Döndürür:
        np.ndarray: Her bireyin fitness değeri, şekil (pop_size,)
    """
    # Her bireyi fitness fonksiyonuna ver, sonuçları topla
    return np.array([fitness_fn(ind) for ind in population])


# Test
pop = initialize_population(pop_size=10, n_dims=2, lower_bound=-5.12, upper_bound=5.12)
fits = evaluate_population(pop, rastrigin)
print("İlk 5 birey:")
for i in range(5):
    print(f"  Birey {i+1}: x={pop[i]}  →  f={fits[i]:.4f}")

In [ ]:
# ============================================================
# SEÇİLİM (SELECTION) — Turnuva Seçimi
# ============================================================

def tournament_selection(population, fitness_values, tournament_size=3):
    """
    Turnuva seçimi: Rastgele k birey seç, en iyisini döndür.
    
    Biyolojik analog: Aynı ortamda rekabet eden bireyler arasında
    en uyumlu olanın hayatta kalması.
    
    Parametreler:
        population      (np.ndarray): Popülasyon matrisi
        fitness_values  (np.ndarray): Her bireyin fitness değerleri
        tournament_size (int)       : Turnuvaya katılan birey sayısı
    
    Döndürür:
        np.ndarray: Seçilen ebeveyn bireyi
    """
    pop_size = len(population)
    
    # Turnuvaya katılacak k bireyin indekslerini rastgele seç
    contestants_idx = np.random.choice(pop_size, size=tournament_size, replace=False)
    
    # Bu bireyler arasında en düşük fitness (minimizasyon) değerine sahip olanı bul
    best_idx = contestants_idx[np.argmin(fitness_values[contestants_idx])]
    
    return population[best_idx].copy()    # Kopyasını döndür (referans değil)


# ============================================================
# ÇAPRAZLAMA (CROSSOVER) — Tek Noktalı
# ============================================================

def single_point_crossover(parent1, parent2, crossover_rate=0.8):
    """
    Tek noktalı çaprazlama: İki ebeveynin genlerini belirli bir noktadan böler.
    
    Biyolojik analog: Kromozomların üreme sırasında birbirleriyle
    genetik materyal alışverişi yapması (rekombinasyon).
    
    Parametreler:
        parent1, parent2 (np.ndarray): İki ebeveyn birey
        crossover_rate   (float)     : Çaprazlama olasılığı
    
    Döndürür:
        tuple: İki yeni çocuk birey
    """
    n_dims = len(parent1)
    
    # crossover_rate olasılığıyla çaprazlama yap
    if np.random.rand() < crossover_rate:
        # Çaprazlama noktası: 1 ile n_dims-1 arasında rastgele
        # (uç noktalar seçilmez — tamamen ebeveynlerden biri olurdu)
        point = np.random.randint(1, n_dims)
        
        # Çocuk 1: Parent1'in [0:point] + Parent2'nin [point:]
        child1 = np.concatenate([parent1[:point], parent2[point:]])
        # Çocuk 2: Parent2'nin [0:point] + Parent1'in [point:]
        child2 = np.concatenate([parent2[:point], parent1[point:]])
    else:
        # Çaprazlama yapılmaz → ebeveynlerin kopyası
        child1, child2 = parent1.copy(), parent2.copy()
    
    return child1, child2


# ============================================================
# MUTASYON (MUTATION) — Gaussian Gürültü
# ============================================================

def gaussian_mutation(individual, mutation_rate=0.1, sigma=0.5,
                       lower=-5.12, upper=5.12):
    """
    Gaussian mutasyon: Her gene belirli olasılıkla Gaussian gürültü ekler.
    
    Biyolojik analog: DNA kopyalanması sırasında oluşan küçük hatalar
    (nokta mutasyonları).
    
    Parametreler:
        individual    (np.ndarray): Mutasyona uğrayacak birey
        mutation_rate (float)     : Her genin mutasyon olasılığı
        sigma         (float)     : Gaussian gürültü standart sapması
        lower, upper  (float)     : Sınır değerler
    
    Döndürür:
        np.ndarray: Mutasyona uğramış birey
    """
    mutated = individual.copy()
    n_dims = len(mutated)
    
    for i in range(n_dims):
        # mutation_rate olasılığıyla bu geni mutasyona uğrat
        if np.random.rand() < mutation_rate:
            # Gaussian (normal) dağılımdan gürültü ekle
            mutated[i] += np.random.normal(0, sigma)
            # Sınır dışına çıkmayı engelle (clip)
            mutated[i] = np.clip(mutated[i], lower, upper)
    
    return mutated


print("✅ GA operatörleri tanımlandı: Selection, Crossover, Mutation")

## 3. Ana GA Döngüsü

In [ ]:
# ============================================================
# ANA GENETİK ALGORİTMA FONKSİYONU
# ============================================================

def genetic_algorithm(
    fitness_fn,          # Optimize edilecek fonksiyon
    n_dims     = 2,      # Problem boyutu
    pop_size   = 100,    # Popülasyon büyüklüğü
    n_gens     = 200,    # Nesil sayısı (iterasyon)
    lower      = -5.12,  # Alt sınır
    upper      =  5.12,  # Üst sınır
    tourn_size = 3,      # Turnuva büyüklüğü
    cx_rate    = 0.8,    # Çaprazlama oranı
    mut_rate   = 0.1,    # Mutasyon oranı
    elitism    = 2       # Elit birey sayısı (en iyileri koru)
):
    """
    Tam Genetik Algoritma implementasyonu.
    
    Döndürür:
        best_solution (np.ndarray): En iyi bulunan çözüm
        best_fitness  (float)     : En iyi fitness değeri
        history       (dict)      : Nesil bazlı istatistikler
    """
    # ── ADIM 1: Başlangıç Popülasyonu ──────────────────────
    population = initialize_population(pop_size, n_dims, lower, upper)
    fitness    = evaluate_population(population, fitness_fn)
    
    # İstatistik kayıt yapısı
    history = {'best': [], 'mean': [], 'std': [], 'populations': []}
    
    # Genel en iyi çözümü takip et
    global_best_idx     = np.argmin(fitness)
    global_best_sol     = population[global_best_idx].copy()
    global_best_fitness = fitness[global_best_idx]
    
    # ── ANA DÖNGÜ ───────────────────────────────────────────
    for gen in range(n_gens):
        
        # ── ELİTİZM: En iyi 'elitism' bireyi doğrudan koru
        # Neden? İyi çözümlerin çaprazlama/mutasyonla bozulmaması için
        elite_indices = np.argsort(fitness)[:elitism]   # En küçük fitness = en iyi
        elites = population[elite_indices].copy()
        
        # ── YENİ POPÜLASYON OLUŞTUR ─────────────────────────
        new_population = []
        
        # Elit bireyler dışında kalanlar için döngü
        while len(new_population) < pop_size - elitism:
            
            # 1. SEÇİLİM: Turnuva seçimiyle iki ebeveyn seç
            parent1 = tournament_selection(population, fitness, tourn_size)
            parent2 = tournament_selection(population, fitness, tourn_size)
            
            # 2. ÇAPRAZLAMA: İki ebeveynden iki çocuk üret
            child1, child2 = single_point_crossover(parent1, parent2, cx_rate)
            
            # 3. MUTASYON: Çocuklara küçük rastgele değişiklikler ekle
            child1 = gaussian_mutation(child1, mut_rate, upper=upper, lower=lower)
            child2 = gaussian_mutation(child2, mut_rate, upper=upper, lower=lower)
            
            new_population.extend([child1, child2])
        
        # Fazla üretilen bireyi kes (pop_size - elitism tam sayı olmayabilir)
        new_population = new_population[:pop_size - elitism]
        
        # Elitleri yeni popülasyona ekle
        population = np.vstack([elites, new_population])
        
        # ── FITNESS DEĞERLENDİRME ───────────────────────────
        fitness = evaluate_population(population, fitness_fn)
        
        # ── EN İYİ ÇÖZÜMÜ GÜNCELLE ─────────────────────────
        gen_best_idx = np.argmin(fitness)
        if fitness[gen_best_idx] < global_best_fitness:
            global_best_fitness = fitness[gen_best_idx]
            global_best_sol     = population[gen_best_idx].copy()
        
        # ── İSTATİSTİK KAYIT ────────────────────────────────
        history['best'].append(global_best_fitness)    # Tüm zamanların en iyisi
        history['mean'].append(np.mean(fitness))        # Neslin ortalama fitness'ı
        history['std'].append(np.std(fitness))          # Neslin çeşitliliği
        
        # İlk 5 nesil ve her 50 neslide bir ilerlemeyi yazdır
        if gen < 5 or (gen + 1) % 50 == 0:
            print(f"Nesil {gen+1:3d} | En iyi: {global_best_fitness:.6f} | "
                  f"Ortalama: {np.mean(fitness):.4f} | Std: {np.std(fitness):.4f}")
    
    return global_best_sol, global_best_fitness, history


# ── ÇALIŞTIR ─────────────────────────────────────────────────
print("🧬 Genetik Algoritma başlatılıyor...\n")
best_sol, best_fit, history = genetic_algorithm(
    fitness_fn = rastrigin,
    n_dims     = 2,
    pop_size   = 100,
    n_gens     = 200
)

print(f"\n{'='*55}")
print(f"✅ Optimizasyon tamamlandı!")
print(f"   En iyi çözüm  : {best_sol}")
print(f"   Fitness değeri: {best_fit:.8f}")
print(f"   Teorik minimum: 0.0")
print(f"   Hata          : {abs(best_fit):.8f}")

## 4. Sonuçların Görselleştirilmesi

In [ ]:
# ============================================================
# KONVERJANs GRAFİĞİ VE ANALİZ
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Genetik Algoritma — Sonuç Analizi', fontsize=15, fontweight='bold')

gens = np.arange(1, len(history['best']) + 1)

# ── Graf 1: Konverjans Eğrisi ──────────────────────────────
axes[0,0].plot(gens, history['best'], 'b-', linewidth=2, label='En iyi fitness')
axes[0,0].plot(gens, history['mean'], 'r--', linewidth=1.5, alpha=0.7, label='Ortalama fitness')
axes[0,0].fill_between(gens,
    np.array(history['mean']) - np.array(history['std']),
    np.array(history['mean']) + np.array(history['std']),
    alpha=0.2, color='red', label='±1 std')
axes[0,0].set_xlabel('Nesil'); axes[0,0].set_ylabel('Fitness Değeri')
axes[0,0].set_title('Konverjans Eğrisi')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)
axes[0,0].set_yscale('log')    # Log ölçeği — gelişmeyi daha iyi gösterir

# ── Graf 2: Çeşitlilik (Diversity) ────────────────────────
axes[0,1].plot(gens, history['std'], 'g-', linewidth=2)
axes[0,1].set_xlabel('Nesil'); axes[0,1].set_ylabel('Standart Sapma')
axes[0,1].set_title('Popülasyon Çeşitliliği')
axes[0,1].grid(True, alpha=0.3)
# Not: Std azalması → popülasyon yakınsıyor (iyi!)
# Std = 0 → erken yakınsama / çeşitlilik kaybı (kötü)
axes[0,1].annotate('Çeşitlilik azalması\n= Yakınsama', 
                    xy=(150, history['std'][149]), fontsize=9,
                    arrowprops=dict(arrowstyle='->', color='black'),
                    xytext=(120, history['std'][149]+0.5))

# ── Graf 3: İyileşme Hızı ─────────────────────────────────
improvement = np.diff(history['best'])    # Nesiller arası değişim
improvement = np.where(improvement < 0, -improvement, 0)  # Sadece iyileşmeler
axes[1,0].bar(gens[1:], improvement, color='steelblue', alpha=0.7)
axes[1,0].set_xlabel('Nesil'); axes[1,0].set_ylabel('İyileşme Miktarı')
axes[1,0].set_title('Nesil Bazlı İyileşme')
axes[1,0].grid(True, alpha=0.3)

# ── Graf 4: Çözüm Konumu ──────────────────────────────────
axes[1,1].contourf(X, Y, Z, levels=50, cmap='viridis', alpha=0.8)
axes[1,1].scatter([0], [0], color='red', s=200, marker='*', 
                   zorder=5, label='Teorik minimum (0,0)')
axes[1,1].scatter([best_sol[0]], [best_sol[1]], color='yellow', 
                   s=200, marker='X', zorder=6, label=f'GA çözümü {best_sol}')
axes[1,1].set_xlabel('x₁'); axes[1,1].set_ylabel('x₂')
axes[1,1].set_title('Bulunan Çözüm Konumu')
axes[1,1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('ga_sonuclar.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hiperparametre Analizi

GA'nın performansı kritik hiperparametrelere bağlıdır. Aşağıda popülasyon büyüklüğünün etkisini analiz ediyoruz.

In [ ]:
# ============================================================
# POPÜLASYON BÜYÜKLÜĞÜ ETKİSİ ANALİZİ
# ============================================================

pop_sizes   = [20, 50, 100, 200]    # Test edilecek popülasyon boyutları
n_runs      = 5                      # Her boyut için kaç kez çalıştır (istatistik için)
results_pop = {}                     # Sonuçları depola

print("Popülasyon büyüklüğü etkisi analiz ediliyor...")
print("(Her kombinasyon 5 kez çalıştırılıyor)\n")

for ps in pop_sizes:
    run_bests = []    # Bu popülasyon boyutu için tüm çalışmaların en iyi değerleri
    
    for run in range(n_runs):
        np.random.seed(run * 10)    # Her çalışma için farklı seed
        
        # GA'yı çalıştır (verbose çıktı olmadan)
        _, best_fit, _ = genetic_algorithm(
            fitness_fn=rastrigin, n_dims=2,
            pop_size=ps, n_gens=100    # Daha hızlı analiz için 100 nesil
        )
        run_bests.append(best_fit)
    
    results_pop[ps] = run_bests
    print(f"Pop={ps:3d}: Ortalama={np.mean(run_bests):.4f} ± {np.std(run_bests):.4f}")


# Box plot ile karşılaştırma
fig, ax = plt.subplots(figsize=(8, 5))

bp = ax.boxplot(
    [results_pop[ps] for ps in pop_sizes],
    labels=[str(ps) for ps in pop_sizes],
    patch_artist=True
)
# Kutuları renklendir
colors = ['#ff9999', '#ffcc99', '#99cc99', '#9999ff']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xlabel('Popülasyon Büyüklüğü')
ax.set_ylabel('En İyi Fitness (5 çalışma)')
ax.set_title('Popülasyon Büyüklüğünün GA Performansına Etkisi')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Teorik optimal')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ga_pop_analiz.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Gözlem: Büyük popülasyon → daha iyi çeşitlilik, ama daha fazla hesaplama")

## 6. Özet ve Değerlendirme

### Avantajlar ✅
- **Gradient gerektirmez** → süreksiz, gürültülü fonksiyonlarda çalışır
- **Global arama** → çoklu arama noktası sayesinde yerel minimumlara daha az takılır
- **Paralel işleme** → popülasyon birbirinden bağımsız hesaplanabilir
- **Esnek kodlama** → binary, gerçek sayılı, permütasyon kodlaması kullanılabilir

### Dezavantajlar ⚠️
- **Yavaş yakınsama** → gradient tabanlı yöntemlere göre daha fazla fonksiyon değerlendirmesi
- **Hiperparametre hassasiyeti** → pop_size, mut_rate, cx_rate ayarlanması zor
- **Erken yakınsama riski** → çeşitlilik kaybıyla yerel minimuma takılabilir
- **Büyük boyutlarda ölçeklenmez** → yüzlerce boyut için yetersiz kalabilir

### Ne Zaman Kullanılmalı?
- Kombinatoryal optimizasyon (TSP, çizelgeleme, rota planlama)
- Mühendislik tasarım optimizasyonu
- Hiperparametre optimizasyonu
- Karmaşık çok-modlu fonksiyonlar